In [0]:
%sql
drop table if exists demo.bronze.customer;
drop table if exists demo.bronze.product;
drop table if exists demo.bronze.order;


In [0]:
from pyspark.sql.functions import monotonically_increasing_id, lit, col, current_date, rand, when

# Load Customer
df_customer = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/Volumes/demo/bronze/data/customers-10000.csv")

# Customer Data clean-up
for col_name in df_customer.columns:
    df_customer = df_customer.withColumnRenamed(col_name, col_name.replace(" ", "_").lower())
#
df_customer.write.mode("overwrite").saveAsTable("demo.bronze.customer")


# Load Product
df_product = spark.read.format("csv").option("header", "true").load("/Volumes/demo/bronze/data/products-10000.csv")

# Product Data clean-up
for col_name in df_product.columns:
    df_product = df_product.withColumnRenamed(col_name, col_name.replace(" ", "_").lower())

# Rename the index column
df_product = df_product.withColumnRenamed("Index", "product_id")
df_product.write.mode("overwrite").saveAsTable("demo.bronze.product")

# Create a customer order table just for testing
df_order = df_customer.crossJoin(df_product).select("customer_id", "product_id", "price")
df_order = df_order.withColumn("order_id", monotonically_increasing_id())
df_order = df_order.withColumn("order_date", current_date())
df_order = df_order.withColumn("order_qty", (rand() * 5).cast("int") + 1)
df_order = df_order.withColumn("order_amount", col("price").cast("float")*col("order_qty").cast("float"))
df_order = df_order.withColumn("order_status",  when(rand() < 0.33, "Shipped").when(rand() < 0.66, "Processed").otherwise("In-Progress"))
df_order.write.mode("overwrite").saveAsTable("demo.bronze.order")

In [0]:
%sql
drop table if exists demo.silver.h_customer;
drop table if exists demo.silver.h_product;
drop table if exists demo.silver.h_order;
drop table if exists demo.silver.s_customer;
drop table if exists demo.silver.s_product;
drop table if exists demo.silver.s_order;


In [0]:
from pyspark.sql.functions import md5, current_timestamp, col, lit

h_cust = spark.table("demo.bronze.customer")

h_cust = h_cust.select(
    "customer_id",
    md5(col("customer_id")).alias("customer_hk"),
    current_timestamp().alias("load_time"),
    lit("system").alias("record_source")
)

h_cust.write.mode("overwrite").saveAsTable("demo.silver.h_customer")

from pyspark.sql.functions import md5, concat_ws

s_customer = h_cust.join(
    spark.table("demo.bronze.customer"),
    on="customer_id",
    how="inner"
).select(
    col("customer_hk"),
    col("customer_id"),
    col("first_name"),
    col("last_name"),
    col("company"),
    col("city"),
    col("country"),
    col("phone_1").alias("phone"),
    col("phone_2").alias("mobile"),
    col("email"),
    md5(concat_ws("||", col("first_name"), col("last_name"), col("company"), col("city"), col("country"), col("phone_1"), col("phone_2"), col("email"))).alias("value_hash")
)

s_customer.write.mode("overwrite").saveAsTable("demo.silver.s_customer")

In [0]:
from pyspark.sql.functions import md5, current_timestamp, col, lit

h_prod = spark.table("demo.bronze.product")
h_prod = h_prod.select(
    "product_id",
    md5(col("product_id")).alias("product_hk"),
    current_timestamp().alias("load_time"),
    lit("system").alias("record_source")
)

h_prod.write.mode("overwrite").saveAsTable("demo.silver.h_product")

from pyspark.sql.functions import concat_ws

s_product = h_prod.join(
    spark.table("demo.bronze.product"),
    on="product_id",
    how="inner"
).select(
    col("product_hk"),
    col("product_id"),
    col("name").alias("product_name"),
    col("description").alias("product_description"),
    col("category").alias("product_category"),
    col("price").alias("product_price"),
    md5(concat_ws("||", col("product_id"), col("product_name"), col("product_category"), col("product_price"))).alias("value_hash")
)

s_product.write.mode("overwrite").saveAsTable("demo.silver.s_product")

# dbutils.fs.cp("/Volumes/workspace/default/mydata/products-10000.csv", "/Volumes/demo/bronze/data")

In [0]:
from pyspark.sql.functions import sha2, concat_ws, md5, sha1, lit, sha2, current_timestamp, col

order = spark.table("demo.bronze.order")
l_order = order.select (
    md5(col("order_id").cast("string")).alias("order_hk"),
    col("order_id"),
    col("customer_id"),
    col("product_id"),
    current_timestamp().alias("load_time"),
    lit("system").alias("record_source")
    )
    
l_order.write.mode("overwrite").saveAsTable("demo.silver.l_order")
display(l_order.limit(10))

s_l_order = order.select(
    md5(col("order_id").cast("string")).alias("order_hk"),
    col("order_id"),
    col("order_date"),
    col("order_amount"),
    col("order_status")
    )
s_l_order = s_l_order.withColumn("remark", lit("Priority Shipping"))
s_l_order = s_l_order.withColumn("load_time", current_timestamp())
s_l_order = s_l_order.withColumn("record_source", lit("system"))
s_l_order = s_l_order.withColumn("value_hash", sha2(concat_ws("|", "order_date", "order_amount", "order_status", "remark", "load_time", "record_source"), 256))

s_l_order.write.mode("overwrite").option("mergeSchema", "true").saveAsTable("demo.silver.s_l_order")

display(s_l_order.limit(10))


In [0]:
l_order = spark.table("demo.silver.l_order")
s_l_order = spark.table("demo.silver.s_l_order")

fact_order = l_order.join(s_l_order, on="order_hk", how="inner")
fact_order = fact_order.select(l_order["order_id"], "customer_id", "product_id", "order_date", "order_amount", "order_status", "remark", l_order["load_time"], l_order["record_source"])
fact_order.write.mode("overwrite").saveAsTable("demo.gold.fact_order")
display(fact_order.limit(10))

h_customer = spark.table("demo.silver.h_customer")
s_customer = spark.table("demo.silver.s_customer")

dim_customer = fact_order.join(h_customer, on="customer_id", how="inner") \
    .join(s_customer, on=["customer_id", "customer_hk"], how="inner") \
    .select(
        "customer_id",
        "first_name",
        "last_name",
        "company",
        "city",
        "country",
        "phone",
        "mobile",
        "email"
    )
dim_customer.write.mode("overwrite").saveAsTable("demo.gold.dim_customer")
display(dim_customer.limit(10))

h_product = spark.table("demo.silver.h_product")
s_product = spark.table("demo.silver.s_product")

dim_product = fact_order.join(h_product, on="product_id", how="inner") \
    .join(s_product, on=["product_id", "product_hk"], how="inner") \
    .select(
        "product_id",
        "product_name",
        "product_description",
        "product_category",
        "product_price"
    )
dim_product.write.mode("overwrite").saveAsTable("demo.gold.dim_product")
display(dim_product.limit(10))

In [0]:
%sql
select * 
from demo.silver.l_order o
inner join demo.silver.s_l_order slo on o.order_hk = slo.order_hk
limit 100